### Fase 3: Consolidação e Curadoria (A Camada Gold)

Nesta etapa, atingimos o ponto culminante da nossa Engenharia de Dados baseada na **Arquitetura Medalhão**: a criação do **Gold Dataset** (Dataset Ouro).

Até o momento, nossos dados estavam separados:

1. Os **textos limpos** das reclamações residem no arquivo da camada Silver (`complaints_lean_2025.parquet`).
  
2. As **predições (sentimentos e scores)** geradas pelo modelo Teacher estão fragmentadas em vários arquivos Parquet na pasta `labeled/`.
  

O objetivo do código a seguir é realizar um **JOIN eficiente** entre essas duas fontes de dados, criando um arquivo único (`training_dataset_2025.parquet`) que servirá como base de estudos para o nosso modelo final (o Student).

**A Regra de Ouro (The Gold Rule): Filtro de Confiança** Para garantir que o modelo Student não aprenda padrões errados, aplicaremos uma rigorosa curadoria de dados diretamente na consulta SQL. Em Machine Learning, a regra é clara: *Garbage In, Garbage Out* (Lixo entra, lixo sai).

Portanto, o código implementa as seguintes regras de negócio:

- **Filtro de Certeza:** Apenas reclamações onde o modelo Teacher teve **75% ou mais de confiança** (`teacher_score >= 0.75`) serão mantidas.
  
- **Binarização Estrita:** Sentimentos classificados como "Neutros" são descartados sumariamente. Manteremos apenas os extremos polares ("Positivo" ou "Negativo"), transformando-os em uma variável binária (`1` e `0`).
  

Para realizar esse cruzamento de milhões de linhas de forma instantânea e sem estourar a memória RAM, utilizaremos o motor analítico do **DuckDB**, que executa a união e a filtragem diretamente sobre os arquivos físicos no disco (*Out-of-Core Processing*).

[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import duckdb
import os
from pathlib import Path

# Caminhos
BASE_DIR = Path.cwd()
LABELED_DIR = BASE_DIR / "data" / "parquet" / "labeled"
TEXT_FILE = BASE_DIR / "data" / "parquet" / "complaints_lean_2025.parquet"
OUTPUT_FILE = BASE_DIR / "data" / "parquet" / "training_dataset_2025.parquet"

def create_gold_dataset():
    print("--- Criando Dataset Final de Treino (Texto + Labels) ---")
    
    if OUTPUT_FILE.exists():
        print(f"[SKIP] O arquivo {OUTPUT_FILE} já existe.")
        return

    con = duckdb.connect()
    
    try:
        # A mágica do DuckDB: JOIN eficiente em disco
        # 1. Lemos todos os arquivos de labels (*.parquet)
        # 2. Juntamos com o arquivo de texto original
        # 3. Filtramos apenas o que o Teacher conseguiu classificar com confiança
        
        query = f"""
            COPY (
                SELECT 
                    T.complaint_id,
                    T.consumer_complaint_narrative AS text,
                    L.teacher_label,
                    L.teacher_score,
                    
                    -- Criar Target Binária (0 ou 1) para o Student
                    CASE 
                        WHEN L.teacher_label = 'Negative' THEN 0
                        WHEN L.teacher_label = 'Positive' THEN 1
                    END AS label_final
                    
                FROM read_parquet('{LABELED_DIR}/*.parquet') AS L
                INNER JOIN read_parquet('{TEXT_FILE}') AS T
                    ON L.complaint_id = T.complaint_id
                
                WHERE 
                    -- FILTRO DE CONFIANÇA (GOLD RULE): Apenas score >= 0.75
                    L.teacher_score >= 0.75
                    
                    -- GARANTIA BINÁRIA: Descarta os neutros já na origem
                    AND L.teacher_label IN ('Negative', 'Positive')
                    
            ) TO '{OUTPUT_FILE}' (FORMAT 'PARQUET', CODEC 'SNAPPY');
        """
        
        print("Executando JOIN e Consolidação...")
        con.sql(query)
        
        # Validação
        count = con.sql(f"SELECT COUNT(*) FROM '{OUTPUT_FILE}'").fetchone()[0]
        print(f"✅ Sucesso! Dataset de Treino criado com {count:,} registros.")
        print(f"📁 Salvo em: {OUTPUT_FILE}")
        
    except Exception as e:
        print(f"❌ Erro: {e}")
        if OUTPUT_FILE.exists():
            os.remove(OUTPUT_FILE)
    finally:
        con.close()

if __name__ == "__main__":
    create_gold_dataset()

--- Criando Dataset Final de Treino (Texto + Labels) ---
Executando JOIN e Consolidação...
✅ Sucesso! Dataset de Treino criado com 484,024 registros.
📁 Salvo em: F:\workspace\postech\datathon_tech_challenge_5\data\parquet\training_dataset_2025.parquet


### Sanity check
Como interpretar os resultados?
Distribuição:

Esperado: Algo como 80-95% Negativo e 5-20% Positivo. Reclamações financeiras são predominantemente ruins.

Alerta Vermelho: Se você tiver 50/50 ou 100% Negativo, algo deu errado no Teacher ou no filtro.

Top Palavras (Top Keywords):

Esperado Negativo: fraud, account, money, bank, charges, stolen.

Esperado Positivo: thank, resolved, help, service, corrected.

Alerta Vermelho: Se a palavra "scam" (fraude) aparecer no Top 5 das Positivas, seu Teacher "alucinou" e rotulou errado.

Confiança:

Esperado: A média deve ser alta (acima de 0.8 ou 0.9). O RoBERTa costuma ser muito convicto.

Alerta: Se a média for 0.55, o modelo não entendeu nada dos textos.

Se essa auditoria passar, você pode ir para o treinamento do Student com a consciência tranquila.

In [2]:
import polars as pl
from collections import Counter
import re
import os

# Caminho do seu Gold Dataset criado no passo anterior
DATASET_PATH = "data/parquet/training_dataset_2025.parquet"

def audit_gold_dataset():
    print(f"--- 🕵️ AUDITORIA DO GOLD DATASET ---")
    
    if not os.path.exists(DATASET_PATH):
        print("❌ Arquivo não encontrado.")
        return

    # Carregar dados
    df = pl.read_parquet(DATASET_PATH)
    total = len(df) # len() do Python funciona sempre
    
    print(f"📂 Total de Registros: {total:,}")
    print("-" * 30)

    # 1. VERIFICAÇÃO DE CLASSES (Balanceamento)
    print("\n📊 1. Distribuição de Classes (label_final):")
    # 0 = Negativo, 1 = Positivo
    # group_by().len() é padrão nas versões novas, mas .count() é mais seguro em antigas
    try:
        dist = df.group_by("label_final").len().sort("label_final")
    except AttributeError:
        dist = df.group_by("label_final").count().sort("label_final") # Fallback
    
    neg = 0
    pos = 0
    
    for row in dist.iter_rows(named=True):
        label = row['label_final']
        # Verifica se a coluna se chama 'len' ou 'count' dependendo da versão
        count = row.get('len', row.get('count'))
        pct = (count / total) * 100
        
        lbl_name = "NEGATIVO (0)" if label == 0 else "POSITIVO (1)"
        if label == -1: lbl_name = "NEUTRO/DESCARTADO"
        
        print(f"   {lbl_name:<15}: {count:,} ({pct:.1f}%)")
        
        if label == 0: neg = count
        if label == 1: pos = count

    # Cálculo de Desbalanceamento
    if pos > 0:
        ratio = neg / pos
        print(f"   >>> Para cada 1 Positivo, existem {ratio:.1f} Negativos.")
    
    # 2. ANÁLISE DE CONFIANÇA (Histograma numérico)
    print("\n🎯 2. Confiança do Teacher (Score):")
    print(df.select(pl.col("teacher_score")).describe())
    
    # CORREÇÃO AQUI: Usamos .height em vez de .len()
    high_conf = df.filter(pl.col("teacher_score") > 0.9).height
    print(f"   Registros com >90% de confiança: {high_conf:,} ({(high_conf/total)*100:.1f}%)")

    # 3. O 'EYE TEST' (Leitura Humana)
    print("\n👀 3. Amostragem Qualitativa (Leia e valide!):")
    
    def print_samples(label_code, nome_classe, n=3):
        print(f"\n   --- Exemplos de {nome_classe} (Alta Confiança) ---")
        samples = df.filter(
            (pl.col("label_final") == label_code) & 
            (pl.col("teacher_score") > 0.9)
        )
        
        # Garante que não tentamos pegar mais amostras do que existem
        n_samples = min(n, samples.height)
        if n_samples == 0:
            print("      (Nenhuma amostra encontrada)")
            return

        samples = samples.sample(n_samples)
        
        for i, row in enumerate(samples.iter_rows(named=True)):
            print(f"   [{i+1}] Score: {row['teacher_score']:.4f}")
            # Trunca texto para não poluir
            print(f"       Texto: {row['text'][:150]}...") 
            print("-" * 10)

    try:
        print_samples(0, "NEGATIVO") # Amostra Negativa
        print_samples(1, "POSITIVO") # Amostra Positiva
    except Exception as e:
        print(f"   (Erro ao gerar amostras: {e})")

    # 4. PALAVRAS-CHAVE (Lógica simples)
    print("\n🔤 4. Top Palavras por Classe (Sanity Check):")
    
    def get_top_words(label_code):
        # Pega todos os textos da classe
        texts_series = df.filter(pl.col("label_final") == label_code).select("text").to_series()
        
        if texts_series.len() == 0:
            return []
            
        texts = texts_series.to_list()
        # Amostra para não demorar
        if len(texts) > 5000: texts = texts[:5000]
        
        all_text = " ".join(texts).lower()
        # Remove pontuação básica
        all_text = re.sub(r'[^\w\s]', '', all_text)
        words = [w for w in all_text.split() if len(w) > 3] # Ignora palavras curtas
        
        return Counter(words).most_common(5)

    print(f"   Top Negativas: {get_top_words(0)}")
    print(f"   Top Positivas: {get_top_words(1)}")
    
    print("\n✅ Fim da Auditoria.")

if __name__ == "__main__":
    audit_gold_dataset()

--- 🕵️ AUDITORIA DO GOLD DATASET ---
📂 Total de Registros: 484,024
------------------------------

📊 1. Distribuição de Classes (label_final):
   NEGATIVO (0)   : 479,242 (99.0%)
   POSITIVO (1)   : 4,782 (1.0%)
   >>> Para cada 1 Positivo, existem 100.2 Negativos.

🎯 2. Confiança do Teacher (Score):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ teacher_score │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 484024.0      │
│ null_count ┆ 0.0           │
│ mean       ┆ 0.834511      │
│ std        ┆ 0.046153      │
│ min        ┆ 0.75          │
│ 25%        ┆ 0.79834       │
│ 50%        ┆ 0.83252       │
│ 75%        ┆ 0.876465      │
│ max        ┆ 0.977051      │
└────────────┴───────────────┘
   Registros com >90% de confiança: 42,337 (8.7%)

👀 3. Amostragem Qualitativa (Leia e valide!):

   --- Exemplos de NEGATIVO (Alta Confiança) ---
   [1] Score: 0.9375
       Texto: {$180.00} was taken plus they put my 


[Voltar para notebook principal](./0_notebook_principal.ipynb)